<a href="https://colab.research.google.com/github/ella941223-cyber/Programming-Language/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai

In [2]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [14]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-flash')

In [4]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CEUaBeqPvLAKqdYnz0KVSe7QNkBYWbNbfs0-VhwvY8c/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [5]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [15]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    model = genai.GenerativeModel('gemini-2.5-flash')

    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = model.generate_content(prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [7]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：98
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 98

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：90
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 90

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：93
已記錄：日期: 2026-03-26, 科目: 英文, 成績: 93

請輸入科目（或輸入 'q' 停止）：q


In [8]:
new_grades

[['2026-03-26', '國文', 98], ['2026-03-26', '數學', 90], ['2026-03-26', '英文', 93]]

In [16]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，全程不對學生成績進行任何評價。\n\n---\n\n### 學生成績摘要與常見迷思整理\n\n**日期：2026年3月26日**\n\n---\n\n#### 一、成績總覽摘要\n\n*   **評量日期：** 2026年3月26日\n*   **涵蓋科目：** 國文、數學、英文\n*   **各科分數：**\n    *   國文：98分\n    *   英文：93分\n    *   數學：90分\n*   **成績分佈觀察：**\n    *   所有科目成績均在90分以上。\n    *   各科分數之間有輕微差異，最高為國文98分，最低為數學90分。\n*   **平均分數：** 約為93.67分（計算方式：(98 + 90 + 93) / 3）\n\n---\n\n#### 二、關於成績的常見迷思整理（不評分，只做總結性說明）\n\n以下是人們在看待學生成績時，常見的一些觀念與迷思，這些僅為一般性觀察，與本次成績單無直接評價關係：\n\n1.  **迷思一：分數能完全代表學習的深度與廣度**\n    *   **說明：** 成績單上的分數通常反映學生在特定時間點、特定評量範圍內，對於教材知識的掌握程度。然而，它可能無法完全捕捉學生在學習過程中的投入程度、批判性思考能力、解決問題的策略、團隊合作能力，或是對科目更深層次的興趣與探究精神。\n    *   **事實：** 分數是一個重要的指標，但它只是學習成果的其中一個切面。\n\n2.  **迷思二：高分代表沒有進步空間，低分代表缺乏能力**\n    *   **說明：** 學習是一個持續的過程，即使是高分學生，也總有機會深化理解、拓展知識應用或學習新的策略。反之，單一評量的分數低，可能受到多種因素影響（如考試壓力、題目類型、當日狀態等），不應直接等同於缺乏學習潛力或該科能力。\n    *   **事實：** 學習與成長是無限的，任何分數都有其背後的成因，並非能力的絕對判斷。\n\n3.  **迷思三：分數高低是衡量個人價值或未來成功的唯一標準**\n    *   **說明：** 成績在學術上具有其意義，但個人的價值和未來的成就，更受到品格、情商、人際關係、創造力、韌性、實際動手能力等多方面綜合因素的影響。過度強調分數可能忽略了這

In [17]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：98
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 98

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：90
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 90

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：93
已記錄：日期: 2026-03-26, 科目: 英文, 成績: 93

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，完全不帶評分判斷。

---

### **學生學習表現摘要**

這份成績列表顯示了學生在2026年3月26日於三個主要科目上的表現。

*   **科目包含：** 國文、數學、英文。
*   **各科成績：**
    *   國文：98分
    *   數學：90分
    *   英文：93分
*   **整體觀察：** 三個科目的成績皆落在90分以上。

---

### **成績相關的常見迷思整理**

在看待學生的成績時，人們常會有一些慣性思維，以下列出幾個常見的迷思，並提供更全面的觀點，以避免單純以分數論斷。

1.  **迷思一：高分代表完全掌握或無需再努力。**
    *   **說明：** 考試成績是當下學習成果的衡量，即使成績優異（如98分），也可能在某些細節、理解深度、應用層面或面對不同題型時仍有進步空間。學習是一個持續的過程，知識領域總有更深廣之處可探索。

2.  **迷思二：成績是衡量學生學習能力或未來成就的唯一標準。**
    *   **說明：** 成績僅反映特定時間點在特定評估方式下的表現，它未能涵蓋學生的多元能力（如批判思考、創造力、解決問題、人際溝通、實踐能力）或學習熱情、性格特質等。許多